# Week 53, Day 2: Memoisation versus tabulation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w053d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from functools import cache

@cache
def f(n):
    return 1 if n == 0 else f(n // 2) + f(n // 3)

print(f(1000), f.cache_info().currsize)

table = [1] + [0] * 1000
for i in range(1, 1001):
    table[i] = table[i // 2] + table[i // 3]
print(table[1000], len(table))

In [ ]:
def fewest_coins(amount, coins):
    INF = float("inf")
    best = [0] + [INF] * amount               # best[a]: fewest coins for amount a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a] = best[a - c] + 1
    return best[amount] if best[amount] < INF else None

print(fewest_coins(6, [1, 3, 4]), fewest_coins(11, [5, 10]), fewest_coins(10_000, [1, 7, 23, 99]))

In [ ]:
def biggest_path(triangle):
    best = list(triangle[-1])                 # the bottom row is its own best
    for r in range(len(triangle) - 2, -1, -1):   # rows from the bottom up
        best = [triangle[r][c] + max(best[c], best[c + 1]) for c in range(r + 1)]
    return best[0]

print(biggest_path([[3], [7, 4], [2, 4, 6], [8, 5, 9, 3]]))

In [ ]:
from functools import cache
import time

def top_down(n):
    @cache
    def f(i):
        return i if i < 2 else (f(i - 1) + f(i - 2)) % 1_000_007
    for i in range(0, n + 1, 200):          # warm the cache in steps to stay shallow
        f(i)
    return f(n)

def bottom_up(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, (a + b) % 1_000_007
    return a

for fn in (top_down, bottom_up):
    start = time.perf_counter()
    answer = fn(200_000)
    print(fn.__name__, answer, f"{time.perf_counter() - start:.3f}s")

In [ ]:
def house_robber(values):
    """Most you can take from houses in a row without taking two neighbours."""
    take, skip = 0, 0                 # best so far if the last house was taken / skipped
    for v in values:
        take, skip = skip + v, max(take, skip)
    return max(take, skip)

print(house_robber([2, 7, 9, 3, 1]), house_robber([5, 1, 1, 5]))

In [ ]:
def fewest_coins_used(amount, coins):
    INF = float("inf")
    best = [0] + [INF] * amount
    last = [None] * (amount + 1)               # last[a]: the coin used last for a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a], last[a] = best[a - c] + 1, c
    if best[amount] == INF:
        return None
    used = []
    while amount:
        used.append(last[amount])
        amount -= last[amount]
    return used

print(fewest_coins_used(6, [1, 3, 4]), fewest_coins_used(63, [1, 5, 10, 21, 25]))

> **Key idea.** Tabulation fills a table from the base cases up, in an order where every entry's ingredients come first: read the recurrence's right-hand side to choose it, and remember that a wrong order fails silently. Memoisation is easier to write and computes only the states it needs; tabulation has no recursion limit, runs faster and lets you save memory by keeping only the rows or values still needed. Rolling away old entries loses the ability to reconstruct the choices, so keep the full table (or a table of choices) when you need them, and walk back from the answer state.

### Exercise: Coins, bottom up

Write `fewest_coins(amount, coins)` by tabulation (no recursion): the fewest coins making the amount exactly, with unlimited coins of each value, or `None` if impossible. Then `count_ways(amount, coins)`: the number of **collections** of coins, by tabulation: loop over the coins on the outside and amounts on the inside, so each collection is counted once.

In [ ]:
def fewest_coins(amount, coins):
    ___

def count_ways(amount, coins):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import cache\ndef test_coins():\n    \"coins\"\n    assert fewest_coins(6, [1, 3, 4]) == 2 and fewest_coins(11, [5, 10]) is None and fewest_coins(0, [2]) == 0\n    assert count_ways(4, [1, 2]) == 3 and count_ways(200, [1, 2, 5, 10, 20, 50, 100, 200]) == 73682\n    assert count_ways(0, [3]) == 1 and count_ways(7, [2, 4]) == 0\n    assert fewest_coins(9_000, [1, 7, 23, 99]) == 96\n    assert count_ways(5_000, [1, 2, 5, 10, 20, 50, 100, 200]) > 10 ** 12")

### Exercise: Houses in a row, and a ring

Write `rob_row(values)` with two rolling variables: the most you can take from houses in a row without taking two neighbours. Then `rob_ring(values)`: the same, but the houses stand in a **circle**, so the first and last are neighbours too. (Hint: either the first house is left out, or the last is.)

In [ ]:
def rob_row(values):
    ___

def rob_ring(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(values, ring):\n    n = len(values)\n    best = 0\n    for picks in itertools.product([0, 1], repeat=n):\n        bad = any(picks[i] and picks[i + 1] for i in range(n - 1))\n        if ring and n > 1 and picks[0] and picks[-1]:\n            bad = True\n        if not bad:\n            best = max(best, sum(v for v, p in zip(values, picks) if p))\n    return best\ndef test_robber():\n    \"rows and rings\"\n    assert rob_row([2, 7, 9, 3, 1]) == 12 and rob_ring([2, 3, 2]) == 3 and rob_ring([1, 2, 3, 1]) == 4\n    assert rob_row([]) == 0 and rob_ring([]) == 0 and rob_ring([7]) == 7\n    rng = random.Random(1)\n    for _ in range(200):\n        vals = [rng.randint(0, 20) for _ in range(rng.randint(0, 10))]\n        assert rob_row(vals) == brute(vals, False) and rob_ring(vals) == brute(vals, True), vals\n    assert rob_row(list(range(100_000))) == sum(range(1, 100_000, 2))")

### Exercise: Fill it backwards

A training course runs sessions in order; session i gives `points[i]`, but taking it means **missing the next** `rest[i]` sessions (you need time to recover). Write `best_points(points, rest)`: the most points. Define f(i) as the best from session i onwards, and fill the table **from the end**. Then `chosen(points, rest)`: the indices taken in one best plan (prefer taking a session on a tie).

In [ ]:
def best_points(points, rest):
    ___

def chosen(points, rest):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(points, rest):\n    n = len(points)\n    best = [0]\n    def go(i, total):\n        if i >= n:\n            best[0] = max(best[0], total)\n            return\n        go(i + 1, total)\n        go(i + 1 + rest[i], total + points[i])\n    go(0, 0)\n    return best[0]\ndef test_backwards():\n    \"fill from the end\"\n    pts, rst = [3, 4, 4, 2], [2, 1, 1, 2]\n    assert best_points(pts, rst) == 6 and chosen(pts, rst) == [1, 3]\n    assert best_points([], []) == 0 and chosen([], []) == []\n    rng = random.Random(2)\n    for _ in range(150):\n        n = rng.randint(0, 10)\n        pts = [rng.randint(0, 9) for _ in range(n)]\n        rst = [rng.randint(0, 3) for _ in range(n)]\n        best = best_points(pts, rst)\n        assert best == brute(pts, rst)\n        picks = chosen(pts, rst)\n        assert sum(pts[i] for i in picks) == best\n        assert all(b > a + rst[a] for a, b in zip(picks, picks[1:]))\n    n = 50_000\n    assert best_points([1] * n, [0] * n) == n")

### Exercise: Two ways, one answer

The number of ways to tile a 1 × n strip with tiles of length 1, 2 and 3. Write `tiles_memo(n)` with `functools.cache` and `tiles_table(n)` with a loop and **three rolling variables** (no list). They must agree; the checks call `tiles_table(100_000) % 1000` too, which the memoised version can't reach.

In [ ]:
from functools import cache

def tiles_memo(n):
    ___

def tiles_table(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_agree():\n    \"two ways\"\n    assert [tiles_table(n) for n in range(8)] == [1, 1, 2, 4, 7, 13, 24, 44]\n    assert all(tiles_memo(n) == tiles_table(n) for n in range(300))\n    a, b, c = 0, 0, 1\n    for _ in range(100_000):\n        a, b, c = b, c, a + b + c\n    assert tiles_table(100_000) == c\n    body = source.split(\"def tiles_table\")[1].split(\"\\ndef \")[0]\n    assert \"[\" not in body, \"rolling variables, not a list\"")

### Exercise: One row at a time

Write `min_path(triangle)` bottom-up with a **single list** that is overwritten row by row: the smallest total from the top to the bottom of a number triangle (moving to c or c + 1 below). Then `min_route(triangle)`: the list of numbers on one smallest path, which needs the **full** table: keep every row's best values and walk down from the top choosing the smaller child (left on a tie).

In [ ]:
def min_path(triangle):
    ___

def min_route(triangle):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef paths(tri):\n    out = []\n    def go(r, c, acc):\n        acc = acc + [tri[r][c]]\n        if r == len(tri) - 1:\n            out.append(acc)\n            return\n        go(r + 1, c, acc)\n        go(r + 1, c + 1, acc)\n    go(0, 0, [])\n    return out\ndef test_min():\n    \"rows\"\n    tri = [[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]\n    assert min_path(tri) == 11 and min_route(tri) == [2, 3, 5, 1]\n    assert min_path([[-10]]) == -10 and min_route([[-10]]) == [-10]\n    rng = random.Random(3)\n    for _ in range(80):\n        tri = [[rng.randint(-5, 9) for _ in range(r + 1)] for r in range(rng.randint(1, 7))]\n        ps = paths(tri)\n        low = min(sum(p) for p in ps)\n        assert min_path(tri) == low and min_route(tri) in ps and sum(min_route(tri)) == low\n        assert min_route(tri) == min(ps, key=sum)\n    big = [[(r * 31 + c * 17) % 50 for c in range(r + 1)] for r in range(1500)]\n    assert min_path(big) == sum(min_route(big))")

### Bug hunt: Filled in the wrong direction

f(i) is the best from house i **onwards**, so it reads f(i + 1) and f(i + 2), but this loop fills i upwards, reading entries that are still 0, and prints `2` instead of `12`. Fill from the end.

In [ ]:
values = [2, 7, 9, 3, 1]
n = len(values)
best = [0] * (n + 2)
for i in range(n):
    best[i] = max(best[i + 1], values[i] + best[i + 2])
print(best[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"dependencies first\"\n    expect_output(\"12\")")

### Bug hunt: Counting orders by mistake

This should count coin **collections** for 4 with coins 1 and 2 (1+1+1+1, 1+1+2, 2+2: three), but with amounts on the outside and coins on the inside it counts every **order** and prints `5`. Swap the loops so coins are on the outside.

In [ ]:
coins = [1, 2]
amount = 4
ways = [1] + [0] * amount
for a in range(1, amount + 1):
    for c in coins:
        if c <= a:
            ways[a] += ways[a - c]
print(ways[amount])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"collections\"\n    expect_output(\"3\")")

### Bug hunt: Rolling too soon

The rolling update of the house-robber variables happens one line at a time, so `skip` is computed from the **new** `take` instead of the old one, and it prints `22`, the total of every house, neighbours and all. Update both at once, with a tuple assignment, so it prints `12`.

In [ ]:
take, skip = 0, 0
for v in [2, 7, 9, 3, 1]:
    take = skip + v
    skip = max(take, skip)
print(max(take, skip))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"simultaneous update\"\n    expect_output(\"12\")")

### Puzzle: Reconstructing the coins

Arrange the lines to fill the fewest-coins table with the last coin used, then walk back. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the fill direction for f(i) = max(f(i + 1), v[i] + f(i + 2)): "up" or "down"
q = ___   # the table after count_ways for amount 4, coins [1, 2] (coins outside), as a list
r = ___   # how many variables a rolling Fibonacci table needs
s = ___   # True or False: memoisation computes every state in the table
t = ___   # rob_row([5, 1, 1, 5])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"down\" and q == [1, 1, 2, 2, 3] and r == 2, \"p, q and r\"\n    assert s is False and t == 10, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the cheapest ticket plan

You'll travel on certain days of the year (a sorted list of distinct days from 1 to 365). A 1-day pass costs `prices[0]`, a 7-day pass `prices[1]` and a 30-day pass `prices[2]`; a pass bought on day d covers d to d + 6 (or d + 29). Write:

- `cheapest(days, prices)`: the least total cost, by tabulation over the days of the year (f(d) = the cheapest way to cover all travel days up to d).
- `plan(days, prices)`: one cheapest list of `(start_day, length)` passes, sorted by start day, found by walking back through the table (prefer the shortest pass on a tie). Start each pass on the first travel day it covers.
- `cheapest_fast(days, prices)`: the same answer with a table indexed by **travel day** (position in the list) instead of calendar day, using `bisect` to find how far each pass reaches, so it works for days up to 10⁹.

In [ ]:
import bisect

def cheapest(days, prices):
    ___

def plan(days, prices):
    ___

def cheapest_fast(days, prices):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(days, prices):\n    best = [None]\n    lengths = (1, 7, 30)\n    def go(i, total):\n        if best[0] is not None and total >= best[0]:\n            return\n        if i == len(days):\n            best[0] = total\n            return\n        for p, L in zip(prices, lengths):\n            j = i\n            while j < len(days) and days[j] < days[i] + L:\n                j += 1\n            go(j, total + p)\n    go(0, 0)\n    return best[0]\ndef covers(days, passes):\n    return all(any(s <= d < s + L for s, L in passes) for d in days)\ndef test_tickets():\n    \"tickets\"\n    assert cheapest([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11\n    assert cheapest([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [2, 7, 15]) == 17\n    assert cheapest([], [2, 7, 15]) == 0 and plan([], [2, 7, 15]) == []\n    p = plan([1, 4, 6, 7, 8, 20], [2, 7, 15])\n    assert p == [(1, 7), (8, 1), (20, 1)], p\n    rng = random.Random(4)\n    for _ in range(80):\n        days = sorted(rng.sample(range(1, 80), rng.randint(0, 12)))\n        prices = sorted(rng.randint(1, 30) for _ in range(3))\n        want = brute(days, prices)\n        assert cheapest(days, prices) == want and cheapest_fast(days, prices) == want\n        pl = plan(days, prices)\n        cost = {1: prices[0], 7: prices[1], 30: prices[2]}\n        assert covers(days, pl) and sum(cost[L] for s, L in pl) == want and pl == sorted(pl)\n        assert all(s in days for s, L in pl)\n    big = sorted(rng.sample(range(1, 10 ** 9), 20_000))\n    assert cheapest_fast(big, [2, 7, 15]) == 40_000")

## Warm-up

Same recurrence, same answer. The memoised version only ever computes the states the question **reaches**: halving and thirding from 1000 visits a few dozen numbers. The table computes all 1,001, in order, because it can't know in advance which ones will be needed. For recurrences that reach far down in single steps, such as `f(n - 1)`, the opposite problem appears: memoisation recurses 1000 calls deep and hits Python's recursion limit, while the loop doesn't care.

## Bottom up

To **tabulate** a recurrence:

1. Make a table with one entry per state (a list for one index, a list of lists for two).
2. Fill in the base cases.
3. Loop over the states in an order where every entry's ingredients are already filled, applying the recurrence.
4. Read off the answer.

The recurrence is exactly yesterday's; only the direction changes. Where memoisation says "to know f(i), first ask for f(i − 1) and f(i − 2)", tabulation says "f(i − 1) and f(i − 2) are already in the table, so f(i) is one line".

## The fill order

The order matters: every entry must come after everything it reads. Look at the recurrence's right-hand side:

- f(i) reads f(i − 1) and f(i − 2): fill i **upwards**.
- f(i) reads f(i + 1) (as in "the best from position i onwards"): fill i **downwards**, from the end.
- f(r, c) reads f(r − 1, c) and f(r, c − 1) (grids, tomorrow and Thursday): fill row by row, left to right.
- f(r, c) reads f(r + 1, ...) (the triangle from yesterday): fill from the **bottom row up**.

A wrong order is a quiet bug: the loop reads entries still holding their starting values (0 or infinity), and the answer is wrong without any error. If you can't find an order (the dependencies go round in a circle), the problem isn't a plain DP: it's a shortest-path problem on a graph, and Dijkstra or Bellman-Ford (Week 51) is the tool.

## Which to choose

Both give the same answers. The differences:

| | Memoisation (top-down) | Tabulation (bottom-up) |
|---|---|---|
| Writing it | Straight from the recurrence | Needs a fill order |
| States computed | Only those the question reaches | Every state in the table |
| Depth | Limited by recursion (about 1000) | No limit |
| Speed | Function calls and hashing cost extra | Plain loops, usually several times faster |
| Memory | The cache holds every state | Often reducible to a few rows |

Use memoisation when the recurrence is awkward to order or most states are never needed (a huge state space where the question touches a few). Use tabulation for long inputs, for speed, and when you want to save memory. A common workflow: write the memoised version first, test it, then tabulate it and check the two agree.

## Rolling arrays

A table entry is only needed while later entries still read it. If f(i) reads only f(i − 1) and f(i − 2), keep **two variables** and slide them along: O(1) memory instead of O(n). If a two-dimensional f(r, c) reads only row r − 1, keep **two rows** (or even one, updated carefully): O(columns) instead of O(rows × columns).

The cost: once old entries are thrown away, you can no longer walk back through them to find **which choices** gave the answer. Keep the full table when you need the choices, and roll it when you only need the number.

## Reading the choices back

To recover **how** the best answer was reached, keep the full table (or a parallel table of the choice made at each state), then start at the answer's state and walk back: at each state, find which option produced its value, record that choice, and move to the state it came from. This is the same idea as rebuilding a path from the `parent` dict in BFS and Dijkstra.

## Practice

## Quiz

## Daily challenge

## Before you go

Take one recurrence from yesterday and write its tabulated version on paper: the table's shape, the base cases, the fill order, and how much of the table you could throw away. Tomorrow: one-dimensional DP in depth, with a catalogue of the patterns that come up again and again.